In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. State (상태 저장소) 정의
class CounterState(TypedDict):
    count: int
    status: str

# 2. Node (작업 함수) 정의
def increment_node(state: CounterState):
    """count를 1 증가시키는 노드"""
    new_count = state["count"] + 1
    return {"count": new_count, "status": "incremented"}

def check_node(state: CounterState):
    """현재 상태를 확인만 하고 그대로 통과시키는 노드"""
    return state 

# 3. 라우팅 함수 (조건부 분기용)
def route_by_count(state: CounterState):
    """count가 3 이상이면 종료, 미만이면 계속 진행하도록 분기 방향을 결정"""
    if state["count"] >= 3:
        return "finish"
    return "continue"

# 4. StateGraph 조립 (Graph-Thinking)
builder = StateGraph(CounterState)

# 노드 등록
builder.add_node("increment", increment_node)
builder.add_node("check", check_node)

# 엣지 연결
builder.add_edge(START, "increment") # 시작 -> increment
builder.add_edge("increment", "check") # increment -> check

# 조건부 엣지 연결 (check 노드 이후 분기)
builder.add_conditional_edges(
    "check",
    route_by_count,
    {
        "continue": "increment", # 다시 돌아감 (Loop)
        "finish": END            # 종료
    }
)

# 5. 그래프 컴파일 및 실행
graph = builder.compile()

# 초기 상태를 주입하여 실행
initial_state = {"count": 0, "status": "start"}
final_state = graph.invoke(initial_state)

print(final_state) 
# 출력 결과: {'count': 3, 'status': 'incremented'}

In [ ]:
# State 정의
class State(TypedDict):
    topic: str
    content: str
    attempt: int
# Node 정의
def write_node(state: State):
    """attempt를 1 증가시키는 노드"""
    new_attempt = state["attempt"] + 1
    topic = state["topic"]
    return {"attempt": new_attempt, "content": f"{topic}에 대한 글입니다 (시도 {new_attempt})"}

def review_node(state: State):
    """입력 받은 그대로 반환하는 단순한 노드"""
    return state

# 라우팅 함수
def route_by_passed(state: State):
    """attempt 횟수가 3이상이면 패스, 미만이면 계속 하도록 하는 노드"""
    if state["attempt"] >= 3:
        return "pass"
    else:
        return "fail"
    
builder  = StateGraph(State)

builder.add_node("write", write_node)
builder.add_node("review", review_node)
